In [1]:
!pip install tensorflow opencv-python scikit-learn numpy matplotlib

   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/350.9 MB ? eta -:--:--
   ---------------------------------------- 0.8/350.9 MB 2.8 MB/s eta 0:02:05
   ---------------------------------------- 1.6/350.9 MB 2.9 MB/s eta 0:02:01
   ---------------------------------------- 2.4/350.9 MB 3.2 MB/s eta 0:01:50
   ---------------------------------------- 3.4/350.9 MB 3.5 MB/s eta 0:01:39
    --------------------------------------- 4.5/350.9 MB 3.8 MB/s eta 0:01:32
    --------------------------------------- 5.5/350.9 MB 4.0 MB/s eta 0:01:26
    --------------------------------------- 6.3/350.9 MB 4.1 MB/s eta 0:01:24
    --------------------------------------- 7.1/350.9 MB 3.9 MB/s eta 0:01:29
    --------------------------------------- 7.9/350.9 MB 3.8 MB/s eta 0:01:30
    --------------------------------------- 8.7/350.9 MB 3.9 MB/s eta 0:01:28
   - -------------------------------------- 9.4/350.9 MB 3.9 MB/s eta 0:01:28


In [7]:
 r'C:\Users\nithi\Downloads\archive (3)\leapGestRecog'

'C:\\Users\\nithi\\Downloads\\archive (3)\\leapGestRecog'

In [8]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


DATASET_PATH = r'C:\Users\nithi\Downloads\archive (3)\leapGestRecog'
IMG_SIZE = 64

images = []
labels = []
label_map = {}
current_label = 0


print("Loading images, please wait...")
for subject in os.listdir(DATASET_PATH):
    subject_path = os.path.join(DATASET_PATH, subject)
    if os.path.isdir(subject_path):
        for gesture in os.listdir(subject_path):
            gesture_path = os.path.join(subject_path, gesture)
            if os.path.isdir(gesture_path):
                if gesture not in label_map:
                    label_map[gesture] = current_label
                    current_label += 1
                
                for img_name in os.listdir(gesture_path):
                    img_path = os.path.join(gesture_path, img_name)
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                        images.append(img)
                        labels.append(label_map[gesture])

X = np.array(images, dtype='float32') / 255.0
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(labels)

print(f"Successfully loaded {len(X)} images across {len(label_map)} classes.")


indices = np.arange(len(X))
np.random.seed(42)
np.random.shuffle(indices)

split_idx = int(len(X) * 0.8)
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(label_map), activation='softmax')
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])


model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))


test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Loading images, please wait...
Successfully loaded 20000 images across 20 classes.


C:\Users\nithi\cat-dog-svm\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 86s 166ms/step - accuracy: 0.9183 - loss: 0.2509 - val_accuracy: 0.9987 - val_loss: 0.0070
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 83s 165ms/step - accuracy: 0.9981 - loss: 0.0072 - val_accuracy: 0.9992 - val_loss: 0.0017
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 83s 166ms/step - accuracy: 0.9993 - loss: 0.0031 - val_accuracy: 0.9995 - val_loss: 0.0012
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 81s 161ms/step - accuracy: 0.9972 - loss: 0.0114 - val_accuracy: 0.9987 - val_loss: 0.0054
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 83s 167ms/step - accuracy: 0.9981 - loss: 0.0061 - val_accuracy: 0.9900 - val_loss: 0.0389
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.9900 - loss: 0.0389
Test Accuracy: 99.00%


In [9]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.9900 - loss: 0.0389
Test Accuracy: 99.00%
